# Exercise 9: CNN


## CIFAR-10 classification using Keras Conv2D

The CIFAR-10 small photo classification problem is a standard dataset used in computer vision and deep learning.


## CIFAR-10 Photo Classification Dataset

CIFAR comes from the Canadian Institute For Advanced Research and the CIFAR-10 dataset was developed along with the CIFAR-100 dataset by researchers at the CIFAR institute. The dataset is comprised of 60,000 32×32 pixel color photographs of objects from 10 classes with following class labels:

0. airplane
1. automobile
2. bird
3. cat
4. deer
5. dog
6. frog
7. horse
8. ship
9. truck

![Alt text](https://student.labranet.jamk.fi/~rmika/images/ciraf-10.png)

It is relatively straightforward to achieve 80% classification accuracy. Top performance on the problem is achieved by deep learning convolutional neural networks with a classification accuracy above 90% on the test dataset.

__This exercises HAVE TO be done in CSC Puhti cloud environment.__

__NOTE: Try to get as good as possible accuracy. Accurary is one of the evalution criteria__.

TOP results will be found from [here](https://student.labranet.jamk.fi/~rmika/DLNN/TOP-scores.htm)



In [ ]:
import os, random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

In [ ]:
# Load CIFAR-10 (from the local copy in ~/data if it exists, otherwise download it)
import pickle

def load_batch(path):
    with open(path, "rb") as f:
        d = pickle.load(f, encoding="bytes")
    x = d[b"data"].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    y = np.array(d[b"labels"]).reshape(-1, 1)
    return x, y

root = os.path.expanduser("~/data/cifar-10-batches-py")
if os.path.isdir(root):
    parts = [load_batch(f"{root}/data_batch_{i}") for i in range(1, 6)]
    x_train_full = np.concatenate([p[0] for p in parts])
    y_train_full = np.concatenate([p[1] for p in parts])
    x_test, y_test = load_batch(f"{root}/test_batch")
else:
    (x_train_full, y_train_full), (x_test, y_test) = cifar10.load_data()
print("Train:", x_train_full.shape, "Test:", x_test.shape)

# Normalise per channel (statistics from the training data only)
x_train_full = x_train_full.astype("float32")
x_test = x_test.astype("float32")
mean = x_train_full.mean(axis=(0, 1, 2))
std = x_train_full.std(axis=(0, 1, 2))
x_train_full = (x_train_full - mean) / std
x_test = (x_test - mean) / std

y_train_full_cat = to_categorical(y_train_full, 10)
y_test_cat = to_categorical(y_test, 10)

# Hold out 5,000 training images as validation set
rng = np.random.RandomState(SEED)
idx = rng.permutation(len(x_train_full))
val_idx, tr_idx = idx[:5000], idx[5000:]
x_val, y_val = x_train_full[val_idx], y_train_full_cat[val_idx]
x_train, y_train = x_train_full[tr_idx], y_train_full_cat[tr_idx]
print(x_train.shape, x_val.shape, x_test.shape)

BATCH_SIZE = 128
AUTOTUNE = tf.data.AUTOTUNE
train_ds = (tf.data.Dataset.from_tensor_slices((x_train, y_train))
            .shuffle(10000, seed=SEED).batch(BATCH_SIZE).prefetch(AUTOTUNE))
val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(256).prefetch(AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test_cat)).batch(256).prefetch(AUTOTUNE)

In [ ]:
# VGG-style CNN: 3 blocks of 2 conv layers (32/64/128 filters) with batch norm and dropout
WEIGHT_DECAY = 5e-4

def conv_bn_relu(x, filters):
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False,
                      kernel_initializer="he_normal",
                      kernel_regularizer=regularizers.l2(WEIGHT_DECAY))(x)
    x = layers.BatchNormalization()(x)
    return layers.Activation("relu")(x)

def conv_block(x, filters, dropout):
    x = conv_bn_relu(x, filters)
    x = conv_bn_relu(x, filters)
    x = layers.MaxPooling2D(2)(x)
    return layers.Dropout(dropout)(x)

def build_model():
    inputs = keras.Input(shape=(32, 32, 3))
    # Data augmentation (only active during training)
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomTranslation(0.1, 0.1, fill_mode="reflect")(x)

    x = conv_block(x, 32, 0.2)
    x = conv_block(x, 64, 0.3)
    x = conv_block(x, 128, 0.4)

    x = layers.Flatten()(x)
    x = layers.Dense(128, use_bias=False, kernel_initializer="he_normal",
                     kernel_regularizer=regularizers.l2(WEIGHT_DECAY))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(10, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="cifar10_vgg_bn_small")

model = build_model()
model.summary()

In [ ]:
# Train: SGD + Nesterov momentum, cosine learning-rate decay, label smoothing.
# The epoch with the best validation accuracy is saved.
EPOCHS = 100
steps_per_epoch = int(np.ceil(len(x_train) / BATCH_SIZE))

lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.05,
    decay_steps=EPOCHS * steps_per_epoch,
    alpha=1e-3)

model.compile(
    optimizer=keras.optimizers.SGD(learning_rate=lr_schedule, momentum=0.9, nesterov=True),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"])

ckpt = keras.callbacks.ModelCheckpoint(
    "best_cifar10.weights.h5", monitor="val_accuracy", mode="max",
    save_best_only=True, save_weights_only=True)

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                    callbacks=[ckpt], verbose=2)

In [ ]:
# Learning curves
h = history.history
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(h["loss"], label="train"); ax[0].plot(h["val_loss"], label="validation")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(h["accuracy"], label="train"); ax[1].plot(h["val_accuracy"], label="validation")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

best_ep = int(np.argmax(h["val_accuracy"]))
print(f"Best epoch: {best_ep + 1}, val acc: {h['val_accuracy'][best_ep]:.4f}")

In [ ]:
# Evaluate the best model on the test set
model.load_weights("best_cifar10.weights.h5")
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Test accuracy: {test_acc:.4f} ({test_acc * 100:.2f} %)  loss: {test_loss:.4f}")

# Test-time augmentation: average the predictions of each image and its mirror image
p = model.predict(x_test, batch_size=256, verbose=0)
p_flip = model.predict(x_test[:, :, ::-1, :], batch_size=256, verbose=0)
y_true = y_test.ravel()
y_pred = (p + p_flip).argmax(1)
print(f"With flip averaging: {np.mean(y_pred == y_true):.4f}")

# Per-class accuracy and the most common confusions
cm = np.zeros((10, 10), dtype=int)
np.add.at(cm, (y_true, y_pred), 1)
print("\nPer-class accuracy:")
for name, a in sorted(zip(CLASS_NAMES, cm.diagonal() / cm.sum(1)), key=lambda t: t[1]):
    print(f"  {name:12s} {a:.3f}")

off = cm.copy(); np.fill_diagonal(off, 0)
print("\nMost common confusions (true -> predicted):")
for k in np.argsort(off.ravel())[::-1][:5]:
    t, q = divmod(k, 10)
    print(f"  {CLASS_NAMES[t]} -> {CLASS_NAMES[q]}: {off[t, q]}")

# Conclusion

## Results
The final CNN reached **89.7 %** test accuracy (**90.4 %** when the predictions for each image and its mirror image are averaged). The best validation accuracy was 90.6 % at epoch 96. Training took 100 epochs, about 25 minutes on 8 CPU cores of Puhti (no GPU), roughly 15 s per epoch.

Training accuracy (91.0 %) and validation accuracy (90.6 %) ended almost equal, so the model did not overfit. The validation curve was noisy in the middle of training and became smooth as the learning rate decayed.

The weakest classes were cat (76 %), bird (82 %) and dog (84 %). They are mostly confused with each other and with frog, cat <-> dog being the largest confusion, which is expected for similar animal classes at only 32x32 pixels. The strongest classes were frog (97.5 %), automobile (96 %) and truck (96 %).

## Development process
I used a VGG-style CNN with three blocks of two 3x3 convolutions (32/64/128 filters), about 0.55 M parameters. It was kept small so that it could be trained on CPU. To reach about 90 % accuracy I combined:
- batch normalization after every convolution,
- progressive dropout (0.2 / 0.3 / 0.4, and 0.5 in the dense layer) and L2 weight decay,
- data augmentation (random horizontal flip and shift),
- label smoothing (0.1),
- SGD with Nesterov momentum and cosine learning-rate decay.

I first ran a 3-epoch test to measure the training speed, then trained for 100 epochs. The best epoch was selected using a separate validation set of 5,000 training images, and the test set was used only once at the end.

## Possible improvements
The curves were still rising slowly when the learning rate reached zero, and the gap between training and validation accuracy was very small. A wider network (for example 64/128/256 filters), ideally trained on a GPU, is therefore the most promising next step. Stronger augmentation (Cutout, MixUp) and an ensemble of several models could also add accuracy.
